# Assignment 3: Preprocessing and modelling historical tropical cyclone records

This assignment works through a whole machine learning workflow on one dataset, a global
collection of tropical cyclone records, the International Best Track Archive for Climate
Stewardship [(IBTrACS)](https://www.ncei.noaa.gov/products/international-best-track-archive).
The column descriptions are given
[here](https://www.ncei.noaa.gov/sites/default/files/2025-09/IBTrACS_v04r01_column_documentation.pdf).
IBTrACS is the World Meteorological Organization's official archive of global tropical cyclone
tracks, assembled from every regional forecast centre. It is real operational data, which means it
is messy: reporting practices differ by basin and have changed over time.

The assignment has two halves. **Parts 1 to 5** are the data exploration and preprocessing work
from Week 2: look at the data, handle missing values, scale, encode, and split. **Parts 6 to 11**
model the cleaned records with both halves of the Week 3 material: **regression** to predict a
continuous intensity, and **classification** to predict a storm's category.

Answer each numbered question in the empty cell below it.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load and aggregate the data set
Using the following code, load in the data set (this takes a few seconds to run: the full archive is a large file):


In [ ]:
url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.ALL.list.v04r01.csv'
df = pd.read_csv(url, parse_dates=['ISO_TIME'], usecols=range(12),
                 skiprows=[1], na_values=[' ', 'NOT_NAMED'],
                 keep_default_na=False, dtype={'NAME': str})

This data set includes cyclone tracks (so it has multiple entries per named cyclone). We'll use the code below to create an aggregated data set for only the named cyclones, which has one entry per cyclone. You will use the data set `dfnamed` for the rest of the assignment.

In [ ]:
dfnamed = df.groupby("NAME").agg(MAX_WIND=('WMO_WIND','max'),
                               MIN_PRES=('WMO_PRES','min'),
                               MEAN_LAT=('LAT','mean'),
                               MEAN_LON=('LON','mean'),
                               BASIN=('BASIN','first'),
                               SUBBASIN=('SUBBASIN','first'),
                               NATURE=('NATURE','first'),
                               SEASON=('SEASON','first')).reset_index()
dfnamed.head()

# these lines of code remove the initial dataframe (we won't need it anymore)
import gc
del df
gc.collect()

## Part 1: Data Exploration

How many named cyclones are there?

In [ ]:
len(dfnamed)

1) Use the `pandas` `hist` method to plot the marginal distributions of the variables in the dataframe `dfnamed`

2) Use the `seaborn` Pairgrid function to create a scatterplot of all of the variables

3) Using `matplotlib`, create a scatter plot of the minimum pressure vs. the maximum wind speed and color by the year the cyclone took place.

## Part 2: Handle missing data
4) How many non-null values does each variable have?

5) Create a new dataframe called `dfdrop` where you have discarded the rows with NaN values in `dfnamed`

6) Create a new dataframe called `dfimputed` where you have imputed the missing values in `dfnamed` with 0.0.

## Part 3: Feature scaling

7) Scale the `MAX_WIND` column of your `dfdrop` dataframe using the StandardScaler and put it into an array called `y`, since it's the target we want to predict 

*Hint*: the `StandardScaler` expects a 2-D input, so reshape your column with `.values.reshape(-1, 1)` before scaling.

8) Create a copy of your `dfdrop` dataframe called `dffeatures`, and drop the `NAME` and `MAX_WIND` columns from your `dffeatures` dataframe, since the `MAX_WIND` variable is going to be your target variable and we won't need the cyclone names any longer

## Part 4: Encode Categorical Variables
9) Check which unique categories each of the variables `BASIN`, `SUBBASIN`, and `NATURE` take, using the `dffeatures` dataframe.

10) Print out the number of cyclones in each basin and subbasin. Also print out how many of each storm type there is.

11) Encode the `BASIN`, `SUBBASIN`, and `NATURE` variables in `dffeatures` using One Hot Encoding, and standardize `MIN_PRES`, `MEAN_LAT`, and `MEAN_LON`, and `SEASON` using the `StandardScaler`.

*Hint*: you can do this in one step using the `sci-kit learn` `ColumnTransformer`. Create an array called `X` that contains the encoded categorical variables and the scaled numerical variables.

12) Print out the feature names associated with the columns in your `X` array.

## Part 5: Train, Validation, & Test Split

13) Split your data set into a training and test/validation data set using the `train_test_split` function with 80% of your original data for training and 20% for the testing and validation data sets.

14) Split your `X_test_val` and `y_test_val` again into separate validation and test data sets, that are 10% each of the original data set. Double check that the size of your final training, validation, and test data sets are correct by printing out the shape of each array.

15) Save the Training, validation, and test data sets and labels as numpy arrays using np.save()

## Modelling the storms

The rest of the assignment uses the cleaned table you built in Part 2. For the modelling parts we
go back to `dfdrop` and build a small feature set by hand from a few of its columns, rather than
the full encoded matrix `X`, so that the fitted coefficients are easy to read and interpret.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_squared_error, r2_score, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)

In [ ]:
# One row per named storm, from the cleaned table built in Part 2.
storms = dfdrop.set_index("NAME")
print(f"{len(storms)} named storms with complete records")
storms.head()

The **Saffir-Simpson scale** classifies storms by maximum sustained wind in knots:

| Category | Wind (kt) |
| --- | --- |
| Tropical storm | 34–63 |
| 1 | 64–82 |
| 2 | 83–95 |
| 3 | 96–112 |
| 4 | 113–136 |
| 5 | 137+ |

Categories 3 and above are **major hurricanes**.

In [ ]:
storms["CATEGORY"] = np.select(
    [storms.MAX_WIND >= 137, storms.MAX_WIND >= 113, storms.MAX_WIND >= 96,
     storms.MAX_WIND >= 83,  storms.MAX_WIND >= 64],
    [5, 4, 3, 2, 1], default=0)          # 0 = tropical storm or weaker

storms["MAJOR"] = (storms.MAX_WIND >= 96).astype(int)

print(storms["CATEGORY"].value_counts().sort_index())
print(f"\nmajor hurricanes: {storms.MAJOR.sum()} of {len(storms)}")

## Part 6: Explore the intensity categories

16) Plot minimum central pressure against maximum wind speed, colored by category. Describe
the relationship you see, and explain physically why it takes that form.

17) How many storms fall in each category? Is this dataset balanced? What does that imply for the classification tasks below?

18) Plot the distribution of maximum wind speed by basin. Do all basins produce the same range of intensities?

## Part 7: Linear regression: predicting intensity

19) Build a feature matrix from `MIN_PRES`, `MEAN_LAT`, `MEAN_LON` and `SEASON`, with
`MAX_WIND` as the target. Split into training and test sets with `random_state=0`.

20) Fit a `LinearRegression` and report the RMSE and $R^2$ on the test set.

21) Report the fitted coefficients. Which feature dominates, and does its sign make physical sense?

22) Plot predicted against observed wind speed for the test set, with a 1:1 line. Where does the model do worst, and is that the part of the range you would most want to get right?

23) Compare against a baseline that always predicts the mean wind speed. How much better is your model? A model that cannot beat this baseline has demonstrated nothing.

## Part 8: Logistic regression, will it be a major hurricane?

24) Using the same features, fit a `LogisticRegression` to predict `MAJOR`. Standardize the
features first. Use `make_pipeline(StandardScaler(), LogisticRegression())`. Why does
scaling matter here when it did not for linear regression?

25) Report accuracy, precision, recall and the confusion matrix on the test set.

26) Report the ROC AUC. Then compare accuracy against a classifier that always predicts the majority class. Which metric is more informative here, and why?

27) Extract the predicted probabilities and plot them against minimum pressure. Where does the model sit near 0.5, and what does that region represent physically?

## Part 9: Softmax regression, the full category scale

28) Fit a multi-class logistic regression to predict `CATEGORY` (six classes). Report the
classification report and a confusion matrix.

29) Which categories does the model confuse most? Explain why in terms of how the Saffir-Simpson categories are defined, pay attention to the width of each wind-speed bin.

30) The category is a deterministic function of `MAX_WIND`, which you predicted in Part 7. Compare two approaches: (a) classify directly, and (b) predict wind speed with regression then apply the category thresholds. Which works better, and why might that be?

## Part 10: Support vector machines

31) Fit an `SVC` with a linear kernel to the `MAJOR` classification task. Compare its
performance to logistic regression.

32) Now fit an `SVC` with an RBF kernel. Does the nonlinear boundary help? Report both.

33) Logistic regression outputs calibrated probabilities; a plain SVM outputs distances from the decision boundary. For a forecaster deciding whether to issue an evacuation order, which is more useful, and why?

## Part 11: Interpretation

34) Minimum pressure is by far the strongest predictor of maximum wind. Both are measurements
*of the same storm at the same time*. If your goal were to **forecast** intensity 24 hours
ahead, would pressure still be available as a feature? What does that tell you about the
difference between a model that explains and a model that predicts?

35) IBTrACS combines reports from different agencies whose practices have changed over decades. Name one way this could bias a model trained on the full record, and suggest how you would check for it.